In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import utulek

utulek.platform.import_globals_notebook(globals())

base_module = utulek


I0000 00:00:1785130913.532450  604391 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785130913.541772  604391 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785130915.373845  604391 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785130915.374222  604391 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


IS_NOTEBOOK_KERNEL_CODE = True
NOTEBOOK_NAME = 2026_07_27-lllm.4.ipynb
ASSET_PATH = /home/gilgamesh/main.syncthing/utulek/experiment/2026_07_27-lllm.ipynb.asset/
Loaded `/home/gilgamesh/main.syncthing/utulek/experiment/.env`.
torch: []
tf: 
jax: [CpuDevice(id=0)]


E0000 00:00:1785130918.457031  604391 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "Qwen/Qwen3.6-35B-A3B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint,
	dtype="auto",
	device_map="auto")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

...

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/693 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [10]:
tokenizer.response_schema = {
	"x-regex":
	r"^(?:(?:<think>)?\s*(?P<thinking>.+?)\s*</think>)?\s*(?:<tool_call>(?P<tool_calls>.*?)\s*</tool_call>)?\s*(?P<content>.+?)?\s*(?:<\|im_end\|>|$)",
	"type": "object",
	"properties": {
	"role": {
	"const": "assistant"
	},
	"content": {
	"type": "string"
	},
	"thinking": {
	"type": "string"
	},
	"tool_calls": {
	"x-regex-iterator": r"^(.*)$",
	"type": "array",
	"items": {
	"type": "object",
	"properties": {
	"type": {
	"const": "function"
	},
	"function": {
	"x-parser": "json",
	"x-parser-args": {
	"allow_non_json": True
	},
	"type": "object",
	"properties": {
	"name": {
	"type": "string"
	},
	"arguments": {
	"type": "object",
	"additionalProperties": {}
	},
	},
	},
	},
	},
	},
	},
}

In [11]:
from transformers import TextIteratorStreamer
from threading import Thread


def prompt(user_prompt: str,
	history=[{
	"role":
	"system",
	"content":
	"You are a general-purpose technical assistant."
	}],
	max_new_tokens: int = 4096,
	enable_thinking: bool = True):
	messages = history
	messages.append({"role": "user", "content": user_prompt})
	input_ids = tokenizer.apply_chat_template(messages,
		add_generation_prompt=True,
		return_dict=True,
		enable_thinking=enable_thinking,
		return_tensors="pt")["input_ids"].to(model.device)
	streamer = TextIteratorStreamer(tokenizer,
		skip_prompt=True,
		skip_special_tokens=True)
	generation_args = {
		"input_ids": input_ids,
		"max_new_tokens": max_new_tokens,
		"temperature": 0.6,
		"top_p": 0.95,
		"top_k": 20,
		"min_p": 0.0,
		# "presence_penalty": 0.0,
		"repetition_penalty": 1.0,
		"do_sample": True,
		"streamer": streamer,
	}
	thread = Thread(target=model.generate,
		kwargs=generation_args)
	thread.start()
	tokens = []
	for token in streamer:
		tokens.append(token)
		yield token
	thread.join()
	history.append({
		"role": "assistant",
		"content": "".join(tokens)
	})


history = [{
	"role":
	"system",
	"content":
	"You are a general-purpose technical assistant."
}]

In [12]:
for token in prompt("Explain Burnside's Lemma.",
	history=history,
	enable_thinking=True,
	max_new_tokens=16384):
	print(token, end="")

Here's a thinking process:

1.  **Understand User Request:** The user wants an explanation of Burnside's Lemma. This is a fundamental result in group theory/combinatorics, specifically in counting 

KeyboardInterrupt: 